# Member 5: Research Questions

Both questions use Member 3's district-by-month table (`outputs/member3_integration/cd_month.csv`): 59 community districts, 2022-01 to 2026-06. **KSI** means people killed or seriously injured. Results are associations, not causes.

In [1]:
import pandas as pd
import plotly.express as px
from scipy.stats import spearmanr

df = pd.read_csv("../outputs/member3_integration/cd_month.csv")

## Research question 1

**In each borough, do months with more planned new housing units also have more KSI people?**

*Why it matters:* if harm rises when building picks up, the city could add safety measures at those times.

*Method:* add up both measures per borough and month, then compute a Spearman correlation per borough. The chart smooths each line over 6 months and scales it so the borough's own average is 100, which puts both lines on one axis.

In [2]:
borough_month = df.groupby(["borough", "year_month"])[["new_units", "ksi"]].sum().reset_index()

rows = []
for borough, g in borough_month.groupby("borough"):
    rho, p = spearmanr(g.new_units, g.ksi)
    rows.append({"borough": borough, "spearman_rho": round(rho, 2), "p_value": round(p, 3)})
display(pd.DataFrame(rows))

for col in ["new_units", "ksi"]:
    smooth = borough_month.groupby("borough")[col].transform(lambda x: x.rolling(6, min_periods=1).mean())
    borough_month[col] = 100 * smooth / borough_month.groupby("borough")[col].transform("mean")

long = borough_month.melt(["borough", "year_month"], var_name="series", value_name="index")
px.line(long, x="year_month", y="index", color="series", facet_col="borough", facet_col_wrap=3,
        title="New housing units and KSI by borough (6-month average, borough mean = 100)").show()

,borough,spearman_rho,p_value
0,BRONX,0.01,0.958
1,BROOKLYN,-0.10,0.478
2,MANHATTAN,0.02,0.904
3,QUEENS,0.22,0.106
4,STATEN ISLAND,0.10,0.455


**Finding:** no link. The correlations run from about -0.10 to 0.22 and none is significant (every p-value is above 0.1). Months with more planned housing are not the months with more serious crashes.

**Limits:** monthly counts are noisy, filings are plans rather than finished buildings, and a missing link here does not prove that building has no effect on safety.

## Research question 2

**Is serious harm packed into a few neighborhoods the way new housing is, or is it more spread out?**

*Why it matters:* it shows whether safety work can focus on a few places.

*Method:* total units and KSI per district, rank districts from most to least, and compare how quickly each cumulative share reaches 50%.

In [3]:
districts = df.groupby("cd_label")[["new_units", "ksi"]].sum()
cumulative = pd.DataFrame({
    col: 100 * districts[col].sort_values(ascending=False).cumsum().reset_index(drop=True) / districts[col].sum()
    for col in districts})
cumulative.index += 1

display(pd.DataFrame({"share held by top 10 (%)": cumulative.loc[10].round(0),
                      "districts needed for half": (cumulative < 50).sum() + 1}))

px.line(cumulative, title="Cumulative share of new units and KSI, districts ranked from most to least",
        labels={"index": "Districts", "value": "Cumulative share (%)", "variable": ""}).add_hline(y=50, line_dash="dot").show()

,share held by top 10 (%),districts needed for half
new_units,42.0,13
ksi,26.0,22


**Finding:** new housing is the more concentrated one. 13 districts hold half of all new units, but it takes 22 districts to reach half of the KSI. The top 10 districts hold 42% of the units and 26% of the KSI.

**Limits:** raw counts, not adjusted for population or road length. Applications are plans, not finished buildings.

**Impact:** serious harm is widespread, so measures aimed at a few districts miss most of it. City-wide measures matter alongside targeting the worst spots.

## Prompts used

**Prompts given to an AI assistant for my part**
* "Let's use Streamlit. Build the simplest, most concise base structure for this project, no complex setups and keep code comments minimal. I'll handle styling later."
* "Here are two research questions I came up with: 1) Do months with more planned housing also have more serious crashes? 2) Is serious harm packed into a few neighborhoods the way new housing is? Propose and generate the best chart for each."
* "I'll rebuild each team member's chart individually so the code aligns directly with what Streamlit expects. Let's do them one by one."
* "Reframe the dashboard headers and text into clear data storytelling prompts using simple, plain English that anyone can understand."
* "Add interactive filters for housing categories and transportation types to the Streamlit app, create a dedicated story section, and generate a minimal README file."
* "Arrange the chart sections for members 1 through 6 side by side in a single row sequence. Improve the overall visual hierarchy of the text and add a light/dark mode toggle."
* "Create a minimal Jupyter notebook for the research questions. Keep the code and markdown strictly concise with zero fluff."